# GLEE Competition agent V7 — conservative policy portfolio

V7 is a score-protection release built from the live evidence rather than a new set of universal constants:

- **Bargaining baseline:** V4, which gained 46.19 rating points over 25 games with 25/25 agreements. A configuration-local quantal-response challenger receives only bounded exploration.
- **Negotiation baseline:** V5, which gained 101.56 points over 55 games with zero fallbacks. The seller policy is frozen; only a small buyer concession challenger may be tested.
- **Persuasion baseline:** V3, the last policy associated with a persuasion increase. V6's credibility scheduler is retired after losing 15.01 points over 35 games. Separate seller-response and buyer-reliability challengers are evaluated conservatively.

An arm is assigned once per game, rewards are harvested only from completed games, and a challenger cannot become the default until it has enough role/configuration-local evidence and its lower confidence bound is no worse than the protected baseline within a small tolerance. Strict V6 action validation remains in place.

No notebook can guarantee a higher rating because opponents, configurations, and percentile scoring change. Run one family at a time with concurrency 1; compare role-stratified payoffs, not only the aggregate rating.


In [ ]:
%pip install -q -U glee-sdk



## API key, imports, and thread-safe memory

Named opponents get cross-game profiles. Hidden opponents get game-local memory.
Stable hashing makes mixed strategies reproducible and concurrency-safe.



In [ ]:
import hashlib
import math
import os
import re
import statistics
import threading
from collections import defaultdict, deque
from getpass import getpass

os.environ["GLEE_API_KEY"] = getpass("GLEE API key: ")

LOCK = threading.RLock()
SEEN = set()
DECISION_LOG = deque(maxlen=1500)

# One arm is fixed for the whole game so terminal rewards have clean credit.
POLICY_ASSIGNMENTS = {}
POLICY_STATS = defaultdict(lambda: {"n": 0, "sum": 0.0, "sum_sq": 0.0})
REWARDED_GAMES = set()
EXPLORATION_RATE = 0.12
MIN_PROMOTION_GAMES = 8
SAFETY_TOLERANCE = 0.03

BASELINE_ARMS = {
    "bargaining": "v4_safe",
    "negotiation": "v5_safe",
    "persuasion": "v3_safe",
}

BARGAINING_MEMORY = defaultdict(lambda: {
    "rejected_floor": 0.0, "opponent_demands": deque(maxlen=40)
})
NEGOTIATION_MEMORY = defaultdict(lambda: {
    "seller_prices": deque(maxlen=50), "buyer_prices": deque(maxlen=50)
})
# Perspective is part of the key. A named opponent's behavior as seller must not
# be contaminated by observations collected while that opponent was the buyer.
PERSUASION_MEMORY = defaultdict(lambda: {
    "pos_high": 0.0, "pos_low": 0.0,
    "neg_high": 0.0, "neg_low": 0.0,
    "positive_buys": 0.0, "positive_decisions": 0.0,
})


## Shared schema helpers



In [ ]:
def clamp(x, low, high):
    return max(low, min(high, x))

def finite_float(value, default=0.0):
    try:
        number = float(value)
        return number if math.isfinite(number) else default
    except (TypeError, ValueError):
        return default

def round_progress(state):
    current = max(1, int(state.get("round", 1)))
    maximum = state.get("max_rounds")
    if state.get("horizon_known") and maximum:
        return clamp((current - 1) / max(1, int(maximum) - 1), 0.0, 1.0)
    # Unknown horizons concede slowly unless explicit repetition is detected.
    return min(0.65, (current - 1) / 16.0)

def final_round(state):
    return bool(state.get("horizon_known") and state.get("max_rounds") and
                int(state.get("round", 1)) >= int(state["max_rounds"]))

def player_index(player):
    return 1 if player in {"player_1", "alice"} else 2

def canonical_player(player):
    return f"player_{player_index(player)}"

def other_player(player):
    return "player_2" if player_index(player) == 1 else "player_1"

def opponent_key(game, family):
    opponent = game.get("opponent") or {}
    if opponent.get("type") != "hidden" and opponent.get("name"):
        return f"{family}:named:{opponent.get('type')}:{opponent['name']}"
    return f"{family}:game:{game.get('game_id', '')}"

def stable_unit(game, salt=""):
    state = game.get("game_state") or {}
    token = f"{game.get('game_id', '')}:{state.get('round', 1)}:{salt}"
    value = int.from_bytes(hashlib.sha256(token.encode()).digest()[:8], "big")
    return value / 2**64

def logistic(x):
    return 1.0 / (1.0 + math.exp(-clamp(x, -60.0, 60.0)))

def allocation(offer, player):
    keys = (("player_1_gain", "alice_gain") if player_index(player) == 1 else
            ("player_2_gain", "bob_gain"))
    for key in keys:
        if key in offer:
            return finite_float(offer[key], None)
    return None

def action_message(text):
    return str(text)[:2000]

def signal_polarity(value):
    if isinstance(value, dict):
        value = value.get("decision", value.get("message"))
    if value is None:
        return None
    text = str(value).strip().lower().replace("_", " ")
    text = re.sub(r"\s+", " ", re.sub(r"[^a-z0-9' ]+", " ", text)).strip()
    negatives = (
        "do not buy", "don't buy", "should not buy", "shouldn't buy",
        "do not recommend", "don't recommend", "recommend against",
        "not worth", "not a good", "not high quality", "low quality",
        "bad product", "buying would be foolish",
    )
    positives = (
        "buy this", "recommend buying", "recommend this", "worth it",
        "high quality", "great product", "good product", "positive",
    )
    if text in {"no", "false", "negative", "not recommended"}:
        return False
    words = set(text.split())
    if any(phrase in text for phrase in negatives) or words & {"pass", "skip", "avoid"}:
        return False
    if text in {"yes", "true", "recommended"}:
        return True
    if text == "buy" or any(phrase in text for phrase in positives):
        return True
    return None

def tail_repeat_count(values, relative_tolerance=0.002):
    if not values:
        return 0
    last = float(values[-1])
    tolerance = max(1e-9, abs(last) * relative_tolerance)
    count = 0
    for value in reversed(values):
        if abs(float(value) - last) <= tolerance:
            count += 1
        else:
            break
    return count


def coarse_bucket(value, edges):
    value = finite_float(value)
    return sum(value >= edge for edge in edges)

def policy_context(game, role):
    state = game["game_state"]
    family = game["game_family"]
    opponent = game.get("opponent") or {}
    opponent_kind = opponent.get("type", "hidden")
    info = "full" if state.get("complete_information") else "hidden"
    horizon = int(state.get("max_rounds", state.get("total_rounds", 0)) or 0)
    horizon_bucket = coarse_bucket(horizon, (5, 10, 25))
    if family == "bargaining":
        own_delta = state.get(f"delta_{player_index(role)}", 0.93)
        return (role, info, horizon_bucket, coarse_bucket(own_delta, (0.8, 0.95, 0.99)), opponent_kind)
    if family == "negotiation":
        value = state.get(f"{canonical_player(game.get('your_player', state.get('current_player')))}_value", 0)
        return (role, info, horizon_bucket, coarse_bucket(value, (25, 75, 150)), opponent_kind)
    p = clamp(finite_float(state.get("p"), 0.5), 0.0, 1.0)
    price = finite_float(state.get("product_price"), 0.0)
    v = finite_float(state.get("v"), max(price, 1.0))
    u = finite_float(state.get("u"), 0.0)
    cutoff = (price - u) / (v - u) if v > u else 1.0
    mode = state.get("seller_message_type") or game.get("valid_actions", {}).get("type", "unknown")
    return (role, str(mode), horizon_bucket, coarse_bucket(p, (0.33, 0.66)),
            coarse_bucket(cutoff, (0.35, 0.65)), opponent_kind)

def candidate_arm(family, role):
    if family == "bargaining":
        return "qre_adaptive"
    if family == "negotiation" and role == "buyer":
        return "buyer_soft"
    if family == "persuasion":
        return "seller_empirical" if role == "seller" else "buyer_lcb"
    return None

def arm_summary(key):
    item = POLICY_STATS[key]
    n = item["n"]
    mean = item["sum"] / n if n else 0.0
    variance = max(0.0, item["sum_sq"] / n - mean * mean) if n else 1.0
    radius = 1.64 * math.sqrt((variance + 0.02) / max(1, n))
    return n, mean, mean - radius, mean + radius

def select_policy_arm(game, role):
    game_id = str(game.get("game_id", "unknown"))
    with LOCK:
        if game_id in POLICY_ASSIGNMENTS:
            return POLICY_ASSIGNMENTS[game_id]["arm"]
        family = game["game_family"]
        baseline = BASELINE_ARMS[family]
        context = policy_context(game, role)
        candidate = candidate_arm(family, role)
        arm = baseline
        # Offline tests always exercise protected policies unless they explicitly
        # install an assignment for a candidate arm.
        is_test = game_id.startswith(("test-", "v5-", "v6-", "v7-", "bad-"))
        if candidate and not is_test:
            base = arm_summary((family, context, baseline))
            trial = arm_summary((family, context, candidate))
            promoted = (trial[0] >= MIN_PROMOTION_GAMES and base[0] >= 4 and
                        trial[2] >= base[2] - SAFETY_TOLERANCE and
                        trial[1] > base[1] + 0.01)
            under_sampled = trial[0] < max(MIN_PROMOTION_GAMES, base[0] // 3)
            explore = (under_sampled and
                       stable_unit(game, f"v7-explore:{role}") < EXPLORATION_RATE)
            if promoted or explore:
                arm = candidate
        POLICY_ASSIGNMENTS[game_id] = {
            "family": family, "role": role, "context": context,
            "arm": arm, "player": canonical_player(game.get(
                "your_player", game["game_state"].get("current_player", "player_1"))),
            "state": dict(game["game_state"]),
        }
        return arm

def _find_numeric(mapping, names):
    if not isinstance(mapping, dict):
        return None
    for name in names:
        value = mapping.get(name)
        if isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value):
            return float(value)
    for value in mapping.values():
        if isinstance(value, dict):
            found = _find_numeric(value, names)
            if found is not None:
                return found
    return None

def normalized_terminal_reward(assignment, payload):
    player = assignment["player"]
    idx = player_index(player)
    payoff = _find_numeric(payload, (
        f"{player}_payoff", f"player_{idx}_payoff", f"payoff_{idx}",
        f"{assignment['role']}_total_payoff", "your_payoff",
        "payoff", "utility", "score",
    ))
    if payoff is None:
        return None
    state = assignment["state"]
    family = assignment["family"]
    if family == "bargaining":
        scale = max(1.0, abs(finite_float(state.get("money_to_divide"), 1.0)))
        return clamp(payoff / scale, -1.0, 1.0)
    if family == "negotiation":
        values = [abs(finite_float(v)) for k, v in state.items() if k.endswith("_value")]
        scale = max([1.0] + values)
        return math.tanh(payoff / scale)
    rounds = max(1, int(state.get("total_rounds", 1) or 1))
    scale = max(1.0, rounds * abs(finite_float(state.get("product_price"), 1.0)))
    return math.tanh(payoff / scale)

def harvest_completed_games(client):
    harvested = []
    for game_id, assignment in list(POLICY_ASSIGNMENTS.items()):
        if game_id in REWARDED_GAMES or game_id.startswith(("test-", "v5-", "v6-", "v7-", "bad-")):
            continue
        try:
            payload = client.game_state(game_id)
            reward = normalized_terminal_reward(assignment, payload)
            if reward is None:
                continue
            key = (assignment["family"], assignment["context"], assignment["arm"])
            with LOCK:
                item = POLICY_STATS[key]
                item["n"] += 1
                item["sum"] += reward
                item["sum_sq"] += reward * reward
                REWARDED_GAMES.add(game_id)
            harvested.append((game_id, assignment["family"], assignment["role"],
                              assignment["arm"], round(reward, 4)))
        except Exception as exc:
            DECISION_LOG.append({"game_id": game_id, "family": assignment["family"],
                                 "action_type": "reward_harvest", "error": repr(exc)})
    return harvested


## 1. Bargaining — protected V4 plus a configuration-local QRE challenger

The protected arm is V4's unshaded acceptance-probability optimization. For responder share \(s\), learned threshold \(\tau\), and width \(w\),

\[
a(s)=\frac{1}{1+\exp(-(s-\tau+0.008)/w)},\qquad
J(s)=a(s)(1-s)^{1.15}-(1-a(s))C_t.
\]

The challenger changes only the threshold shade, payoff curvature, and Alice failure cost. It is evaluated within role, information, discount, horizon, and opponent-type buckets; V4 remains the default until conservative promotion criteria are met.


In [ ]:
def player_delta(state, player, default=0.93):
    return clamp(finite_float(state.get(f"delta_{player_index(player)}", default), default),
                 0.001, 0.9999)

def rubinstein_responder_share(proposer_delta, responder_delta):
    denominator = 1.0 - proposer_delta * responder_delta
    proposer_share = ((1.0 - responder_delta) / denominator
                      if denominator > 1e-12 else 0.5)
    return clamp(1.0 - proposer_share, 0.001, 0.999)

def bargaining_offer_series(state, money):
    series = {"player_1": [], "player_2": []}
    if money <= 0:
        return series
    for record in state.get("history", []):
        if not isinstance(record, dict):
            continue
        offer = record.get("offer") or {}
        proposer = canonical_player(record.get("proposer", offer.get("proposer", "player_1")))
        demand = allocation(offer, proposer)
        if demand is not None:
            series[proposer].append(clamp(demand / money, 0.0, 1.0))
    return series

def bargaining_stall_count(state, money):
    series = bargaining_offer_series(state, money)
    return min(tail_repeat_count(series["player_1"], 0.003),
               tail_repeat_count(series["player_2"], 0.003))

def bargaining_profile_key(game, state, me):
    # Reservation shares are configuration dependent. Never carry a hard
    # rejection floor across roles or incompatible discount conditions.
    info = "complete" if state.get("complete_information") else "hidden"
    own_delta = round(player_delta(state, me), 3)
    opponent = other_player(me)
    visible_opponent_delta = state.get(f"delta_{player_index(opponent)}")
    opponent_delta = (round(finite_float(visible_opponent_delta), 3)
                      if visible_opponent_delta is not None else "x")
    family = (f"bargaining:{canonical_player(me)}:{info}:"
              f"d{own_delta}:od{opponent_delta}")
    return opponent_key(game, family)

def update_bargaining_memory(game, me, opponent, money):
    key = bargaining_profile_key(game, game["game_state"], me)
    with LOCK:
        model = BARGAINING_MEMORY[key]
        for record in game["game_state"].get("history", []):
            if not isinstance(record, dict):
                continue
            offer = record.get("offer") or {}
            proposer = canonical_player(record.get("proposer", offer.get("proposer", "player_1")))
            decision = record.get("decision")
            if isinstance(decision, dict):
                decision = decision.get("decision")
            event = ("bargaining-v7", game.get("game_id"), record.get("round"),
                     proposer, str(decision),
                     tuple(sorted((str(k), str(v)) for k, v in offer.items())))
            if event in SEEN:
                continue
            SEEN.add(event)
            if proposer == canonical_player(me) and str(decision).lower() == "reject":
                rejected = allocation(offer, opponent)
                if rejected is not None and money > 0:
                    model["rejected_floor"] = max(model["rejected_floor"], rejected / money)
            if proposer == canonical_player(opponent):
                demand = allocation(offer, opponent)
                if demand is not None and money > 0:
                    model["opponent_demands"].append(clamp(demand / money, 0.0, 1.0))
        return {"rejected_floor": model["rejected_floor"],
                "opponent_demands": list(model["opponent_demands"])}

def estimate_bargaining_floor(game, state, me, opponent, model):
    t = round_progress(state)
    if state.get("complete_information"):
        prior = rubinstein_responder_share(player_delta(state, me),
                                           player_delta(state, opponent))
    else:
        opponent_type = (game.get("opponent") or {}).get("type")
        prior = (0.46 if opponent_type == "human" else 0.43) + 0.02 * t
    evidence = model["rejected_floor"] + 0.006 if model["rejected_floor"] else 0.0
    if model["opponent_demands"]:
        recent = model["opponent_demands"][-5:]
        demand_floor = statistics.median(recent) - (0.065 - 0.020 * t)
        evidence = max(evidence, demand_floor)
    return clamp(max(prior, evidence), 0.20, 0.999)

def bargaining_strategy(game):
    state = game["game_state"]
    me = canonical_player(game.get("your_player", state["current_player"]))
    opponent = other_player(me)
    money = finite_float(state["money_to_divide"])
    t = round_progress(state)
    my_delta = player_delta(state, me)
    model = update_bargaining_memory(game, me, opponent, money)
    floor = estimate_bargaining_floor(game, state, me, opponent, model)
    stalls = bargaining_stall_count(state, money)
    arm = select_policy_arm(game, me)

    if game["valid_actions"]["type"] == "offer":
        if stalls >= 3 and model["opponent_demands"]:
            # Match the opponent's revealed demand rather than repeating a split
            # they have already rejected indefinitely.
            responder_share = clamp(model["opponent_demands"][-1], 0.20, 0.9999)
        else:
            best = None
            alice = player_index(me) == 1
            if arm == "qre_adaptive":
                shade = 0.018 if alice else 0.006
                payoff_power = 1.22 if alice else 1.15
                failure_multiplier = 0.40 if alice else 1.0
            else:  # exact V4 policy constants
                shade, payoff_power, failure_multiplier = 0.0, 1.15, 1.0
            search_floor = clamp(floor - shade, 0.18, 0.999)
            failure_cost = ((0.04 + 0.24 * t + 0.55 * (1.0 - my_delta)) *
                            failure_multiplier)
            # 10% through 99.5% in half-percentage-point increments.
            for step in range(20, 200):
                responder_share = step / 200.0
                width = 0.012 if search_floor > 0.80 else 0.022
                probability = logistic((responder_share - search_floor + 0.008) / width)
                own_share = 1.0 - responder_share
                objective = probability * own_share**payoff_power - (1.0 - probability) * failure_cost
                candidate = (objective, own_share, responder_share)
                if best is None or candidate > best:
                    best = candidate
            responder_share = best[2]
        responder_gain = round(money * responder_share, 8)
        own_gain = money - responder_gain
        action = ({"alice_gain": own_gain, "bob_gain": responder_gain}
                  if player_index(me) == 1 else
                  {"alice_gain": responder_gain, "bob_gain": own_gain})
        if state.get("messages_allowed"):
            pct = round(100 * responder_share, 1)
            action["message"] = action_message(
                f"I offer you {pct}% now, accounting for discounting and the observed negotiation path."
            )
        return action

    current_gain = allocation(state.get("last_offer") or {}, me)
    if current_gain is None:
        return {"decision": "reject"}
    if final_round(state):
        return {"decision": "accept" if current_gain >= 0 else "reject"}
    if stalls >= 3 and current_gain > 0:
        return {"decision": "accept"}
    next_own_share = 1.0 - floor
    deal_probability = clamp(0.80 + 0.10 * t - 0.20 * model["rejected_floor"], 0.45, 0.92)
    continuation_share = my_delta * next_own_share * deal_probability
    role_margin = (0.025 * (1.0 - t)
                   if arm == "qre_adaptive" and player_index(me) == 1 and stalls == 0
                   else 0.0)
    risk_floor = max(0.0, 0.34 + role_margin - 0.08 * t - 0.07 * stalls)
    required = money * max(risk_floor, continuation_share)
    return {"decision": "accept" if current_gain + 1e-9 >= required else "reject"}


## 2. Negotiation — frozen V5 core with bounded buyer residual

V5 remains the protected arm. With complete information and surplus \(S=v_b-v_s\), its target is

\[
P_t=\begin{cases}
v_s+c_tS,&\text{seller},\\
v_b-c_tS,&\text{buyer},
\end{cases}
\quad
c_t=\begin{cases}0.74-0.14t,&\text{seller},\\0.66-0.10t,&\text{buyer}.\end{cases}
\]

The seller arm is frozen. A buyer-only challenger reduces requested capture by at most four percentage points and slightly lowers continuation utility. It cannot be promoted without completed-game evidence in the same role/configuration bucket.


In [ ]:
def offer_sender(item, record, field):
    sender = item.get("from_player") if isinstance(item, dict) else None
    if sender:
        return canonical_player(sender)
    if field == "counteroffer" and record.get("decided_by"):
        return canonical_player(record["decided_by"])
    return None

def negotiation_price_series(state):
    series = {"player_1": [], "player_2": []}
    for record in state.get("history", []):
        if not isinstance(record, dict):
            continue
        for field in ("offer", "counteroffer"):
            item = record.get(field)
            if isinstance(item, dict) and item.get("price") is not None:
                sender = offer_sender(item, record, field)
                if sender:
                    series[sender].append(finite_float(item["price"]))
    return series

def negotiation_stall_count(state, me, opponent):
    series = negotiation_price_series(state)
    return min(tail_repeat_count(series[canonical_player(me)], 0.001),
               tail_repeat_count(series[canonical_player(opponent)], 0.001))

def negotiation_profile_key(game, state):
    me = canonical_player(game.get("your_player", state["current_player"]))
    role = state.get(f"{me}_role", "unknown")
    info = "complete" if state.get("complete_information") else "hidden"
    return opponent_key(game, f"negotiation:{role}:{info}")

def update_negotiation_memory(game, state):
    key = negotiation_profile_key(game, state)
    with LOCK:
        model = NEGOTIATION_MEMORY[key]
        for record in state.get("history", []):
            if not isinstance(record, dict):
                continue
            for field in ("offer", "counteroffer"):
                item = record.get(field)
                if not isinstance(item, dict) or item.get("price") is None:
                    continue
                sender = offer_sender(item, record, field)
                if sender is None:
                    continue
                price = finite_float(item["price"])
                event = ("negotiation-v7", game.get("game_id"), record.get("round"),
                         field, sender, price)
                if event in SEEN:
                    continue
                SEEN.add(event)
                role = state.get(f"{sender}_role")
                if role in {"seller", "buyer"}:
                    model[f"{role}_prices"].append(price)
        return {name: list(values) for name, values in model.items()}

def opponent_prices_in_game(state, opponent):
    return negotiation_price_series(state)[canonical_player(opponent)]

def projected_opponent_price(prices, role):
    if not prices:
        return None
    latest = prices[-1]
    if len(prices) < 2:
        return latest
    step = latest - prices[-2]
    # Only extrapolate concessions in the economically expected direction.
    if role == "seller":
        step = min(0.0, step)
    else:
        step = max(0.0, step)
    return max(0.0, latest + 0.6 * step)

def negotiation_strategy(game):
    state = game["game_state"]
    me = canonical_player(game.get("your_player", state["current_player"]))
    opponent = other_player(me)
    role = state[f"{me}_role"]
    arm = select_policy_arm(game, role)
    opponent_role = state[f"{opponent}_role"]
    my_value = finite_float(state[f"{me}_value"])
    t = round_progress(state)
    update_negotiation_memory(game, state)
    observed = opponent_prices_in_game(state, opponent)
    stalls = negotiation_stall_count(state, me, opponent)
    opponent_value = state.get(f"{opponent}_value")
    surplus = None

    if state.get("complete_information") and opponent_value is not None:
        opponent_value = finite_float(opponent_value)
        seller_value = my_value if role == "seller" else opponent_value
        buyer_value = my_value if role == "buyer" else opponent_value
        surplus = buyer_value - seller_value
        own_capture = ((0.74 - 0.14 * t) if role == "seller" else
                       ((0.62 - 0.08 * t) if arm == "buyer_soft" else
                        (0.66 - 0.10 * t)))
        if observed and surplus > 1e-12:
            last_price = observed[-1]
            opponent_demand = ((last_price - seller_value) / surplus if opponent_role == "seller"
                               else (buyer_value - last_price) / surplus)
            feasible_capture = 1.0 - clamp(opponent_demand - (0.05 + 0.04 * t), 0.0, 1.0)
            own_capture = 0.58 * own_capture + 0.42 * feasible_capture
        own_capture = clamp(own_capture, 0.52, 0.82)
        if surplus <= 0:
            target = my_value
        elif role == "seller":
            target = seller_value + own_capture * surplus
        else:
            target = buyer_value - own_capture * surplus
    elif observed:
        anchor = projected_opponent_price(observed, opponent_role)
        claim = ((0.70 - 0.12 * t) if role == "seller" else
                 ((0.58 - 0.08 * t) if arm == "buyer_soft" else
                  (0.62 - 0.10 * t)))
        if role == "seller" and anchor >= my_value:
            target = my_value + claim * (anchor - my_value)
        elif role == "buyer" and anchor <= my_value:
            target = my_value - claim * (my_value - anchor)
        else:
            target = my_value
    elif role == "seller":
        target = my_value * (1.36 - 0.16 * t)
    else:
        target = my_value * ((0.82 + 0.08 * t) if arm == "buyer_soft" else
                             (0.80 + 0.10 * t))

    target = max(0.0, finite_float(target, my_value))
    if game["valid_actions"]["type"] == "offer":
        action = {"product_price": round(target, 8)}
        if state.get("messages_allowed"):
            action["message"] = action_message(
                "This price is inside the feasible interval revealed by our offers."
            )
        return action

    price = finite_float((state.get("last_offer") or {}).get("price"), my_value)
    offered_utility = price - my_value if role == "seller" else my_value - price
    profitable = offered_utility >= -1e-9
    if final_round(state):
        return {"decision": "AcceptOffer" if profitable else "RejectOffer"}
    if profitable and stalls >= 2:
        return {"decision": "AcceptOffer"}
    if not profitable and stalls >= 3 and not state.get("horizon_known"):
        return {"decision": "WalkAway"}

    target_utility = abs(target - my_value)
    offered_capture = offered_utility / surplus if surplus is not None and surplus > 1e-12 else None
    continuation_base = (0.80 if role == "seller" else
                         (0.72 if arm == "buyer_soft" else 0.75))
    continuation = target_utility * (continuation_base - 0.12 * t -
                                     0.08 * min(stalls, 2))
    if profitable and (offered_utility + 1e-9 >= continuation or
                       (offered_capture is not None and offered_capture >= 0.50 + 0.05 * (1 - t))):
        return {"decision": "AcceptOffer"}

    blend = 0.24 + 0.43 * t + 0.08 * min(stalls, 2)
    blend = clamp(blend, 0.0, 0.78)
    counter = (1.0 - blend) * target + blend * price
    counter = max(my_value, counter) if role == "seller" else min(my_value, counter)
    action = {"decision": "RejectOffer", "product_price": round(max(0.0, counter), 8)}
    if state.get("messages_allowed"):
        action["message"] = action_message(
            "I am conceding inside my individually rational range."
        )
    return action


## 3. Persuasion — protected V3 precision policy plus role-specific challengers

The protected buyer uses V3's directly smoothed signal precision,

\[
\widehat\rho_m=\frac{4\rho_{m,0}+n_{m,H}}{4+n_{m,H}+n_{m,L}},\qquad
\mathbb E[V\mid m]=\widehat\rho_m v+(1-\widehat\rho_m)u,
\]

and buys when expected value plus V3's finite-horizon information bonus reaches price. The protected seller uses V3's credibility-constrained pooling rate. The buyer challenger replaces the posterior mean by a one-sided lower confidence estimate; the seller challenger estimates whether positive recommendations actually induce purchases and scales pooling down for empirically skeptical receivers. Seller and buyer memories remain separated.


In [ ]:
def persuasion_profile_key(game, seller_view):
    state = game["game_state"]
    perspective = "buyer-response" if seller_view else "seller-reliability"
    action_type = game.get("valid_actions", {}).get("type", "")
    mode = state.get("seller_message_type") or (
        "text" if action_type == "seller_message" else "binary"
    )
    return opponent_key(game, f"persuasion:v7:{perspective}:{mode}")

def update_persuasion_memory(game, seller_view):
    state = game["game_state"]
    key = persuasion_profile_key(game, seller_view)
    with LOCK:
        model = PERSUASION_MEMORY[key]
        for record in state.get("history", []):
            if not isinstance(record, dict):
                continue
            signal = signal_polarity(record.get("seller_message"))
            quality = record.get("quality")
            decision = record.get("buyer_decision")
            if isinstance(decision, dict):
                decision = decision.get("decision")
            bought = record.get("bought") is True or str(decision).lower() == "yes"
            event = ("persuasion-v7", "seller" if seller_view else "buyer",
                     game.get("game_id"), record.get("round"), signal,
                     quality if (seller_view or bought) else "unobserved", str(decision))
            if event in SEEN:
                continue
            SEEN.add(event)
            # The buyer sees quality only after purchase; the seller sees it always.
            if signal is not None and quality in {"high", "low"} and (seller_view or bought):
                model[f"{'pos' if signal else 'neg'}_{quality}"] += 1.0
            if seller_view and signal is True and str(decision).lower() in {"yes", "no"}:
                model["positive_decisions"] += 1.0
                model["positive_buys"] += float(str(decision).lower() == "yes")
        return dict(model)

def strategic_signal_prior(positive, p):
    q_high, q_low = ((0.90, 0.24) if positive else (0.10, 0.76))
    denominator = p * q_high + (1.0 - p) * q_low
    return p * q_high / denominator if denominator > 1e-12 else p

def smoothed_signal_precision(model, positive, p):
    high = model["pos_high"] if positive else model["neg_high"]
    low = model["pos_low"] if positive else model["neg_low"]
    prior = strategic_signal_prior(positive, p)
    return (4.0 * prior + high) / (4.0 + high + low)

def precision_lower_bound(model, positive, p):
    high = model["pos_high"] if positive else model["neg_high"]
    low = model["pos_low"] if positive else model["neg_low"]
    prior = strategic_signal_prior(positive, p)
    alpha = 4.0 * prior + high
    beta = 4.0 * (1.0 - prior) + low
    mean = alpha / (alpha + beta)
    variance = alpha * beta / ((alpha + beta) ** 2 * (alpha + beta + 1.0))
    # Mild one-sided caution; the challenge is deliberately bounded.
    return clamp(mean - 0.55 * math.sqrt(max(0.0, variance)), 0.0, 1.0)

def v3_pool_probability(game, state, model, p, price, t, round_no, total_rounds):
    if "v" not in state or "u" not in state:
        return 0.0
    v, u = finite_float(state["v"]), finite_float(state["u"])
    if price <= u:
        return 1.0
    if price >= v or not (0.0 < p < 1.0) or v <= u:
        return 0.0
    cutoff = clamp((price - u) / (v - u), 1e-6, 1.0 - 1e-6)
    static_pool = p * (1.0 - cutoff) / (cutoff * (1.0 - p))
    response_rate = ((model["positive_buys"] + 1.5) /
                     (model["positive_decisions"] + 2.0))
    ramp = 0.12 + 0.88 * t**1.65
    pool_probability = clamp(static_pool * ramp * (0.55 + 0.60 * response_rate), 0.0, 1.0)
    precision_after_lie = ((2.5 + model["pos_high"]) /
                           (2.5 + 0.8 + model["pos_high"] + model["pos_low"] + 1.0))
    credibility_ok = precision_after_lie >= min(0.97, cutoff + 0.025)
    truthful_prefix_done = round_no > max(1, math.ceil(0.12 * total_rounds))
    return pool_probability if truthful_prefix_done and credibility_ok else 0.0

def persuasion_strategy(game):
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    price = finite_float(state["product_price"])
    p = clamp(finite_float(state.get("p"), 0.5), 0.0, 1.0)
    round_no = max(1, int(state.get("round", 1)))
    total_rounds = max(round_no, int(state.get("total_rounds", round_no)))
    t = (round_no - 1) / max(1, total_rounds - 1)
    seller_view = action_type in {"seller_message", "seller_recommendation"}
    role = "seller" if seller_view else "buyer"
    arm = select_policy_arm(game, role)
    model = update_persuasion_memory(game, seller_view)

    if seller_view:
        quality = state.get("current_quality")
        recommend = quality == "high"
        if quality == "low":
            probability = v3_pool_probability(game, state, model, p, price, t,
                                               round_no, total_rounds)
            if arm == "seller_empirical":
                response_rate = ((model["positive_buys"] + 1.5) /
                                 (model["positive_decisions"] + 2.0))
                # Scale down against skeptical receivers; never exceed V3.
                probability *= clamp((response_rate - 0.20) / 0.55, 0.0, 1.0)
            recommend = stable_unit(game, f"v7-pool:{arm}") < probability
        if action_type == "seller_recommendation":
            return {"decision": "yes" if recommend else "no"}
        return {"message": action_message(
            "My recommendation: BUY this product." if recommend else
            "My recommendation: PASS on this product this round."
        )}

    v, u = finite_float(state["v"]), finite_float(state["u"])
    if price <= u:
        return {"decision": "yes"}
    if price > v:
        return {"decision": "no"}
    signal = signal_polarity(state.get("seller_message"))
    if signal is None:
        posterior = p
    elif arm == "buyer_lcb":
        posterior = precision_lower_bound(model, signal, p)
    else:
        posterior = smoothed_signal_precision(model, signal, p)
    expected_value = posterior * v + (1.0 - posterior) * u
    observations = (model["pos_high"] + model["pos_low"] if signal is True else
                    model["neg_high"] + model["neg_low"] if signal is False else 0.0)
    remaining_fraction = (total_rounds - round_no) / max(1, total_rounds)
    information_bonus = (0.012 * max(0.0, v - u) * remaining_fraction /
                         math.sqrt(1.0 + observations) if signal is True else 0.0)
    return {"decision": "yes" if expected_value + information_bonus >= price else "no"}


## Validated dispatcher and safe fallback

Invalid moves and turn timeouts are scored at the fifth percentile. Every V3 move is
therefore validated locally; unexpected schemas fall back to a conservative legal
action and are recorded in `DECISION_LOG`.



In [ ]:
STRATEGIES = {
    "bargaining": bargaining_strategy,
    "negotiation": negotiation_strategy,
    "persuasion": persuasion_strategy,
}

def fallback_action(game):
    family = game["game_family"]
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    if family == "bargaining":
        if action_type == "offer":
            money = finite_float(state["money_to_divide"])
            alice = round(money / 2.0, 8)
            return {"alice_gain": alice, "bob_gain": money - alice}
        return {"decision": "accept"}
    if family == "negotiation":
        me = canonical_player(game.get("your_player", state["current_player"]))
        role = state[f"{me}_role"]
        value = finite_float(state[f"{me}_value"])
        if action_type == "offer":
            return {"product_price": max(0.0, value)}
        price = finite_float((state.get("last_offer") or {}).get("price"), value)
        profitable = price >= value if role == "seller" else price <= value
        if profitable:
            return {"decision": "AcceptOffer"}
        if final_round(state):
            return {"decision": "RejectOffer"}
        return {"decision": "RejectOffer", "product_price": max(0.0, value)}
    if action_type == "seller_message":
        return {"message": "My recommendation: PASS this round."}
    if action_type == "seller_recommendation":
        return {"decision": "no"}
    p = finite_float(state.get("p"), 0.5)
    expected = p * finite_float(state.get("v")) + (1 - p) * finite_float(state.get("u"))
    return {"decision": "yes" if expected >= finite_float(state["product_price"]) else "no"}

def is_finite_number(value):
    return (isinstance(value, (int, float)) and not isinstance(value, bool) and
            math.isfinite(float(value)))

def contract_action_keys(game, action):
    family = game["game_family"]
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    if family == "bargaining":
        keys = {"alice_gain", "bob_gain"} if action_type == "offer" else {"decision"}
    elif family == "negotiation":
        if action_type == "offer":
            keys = {"product_price"}
        else:
            keys = {"decision"}
            if action.get("decision") == "RejectOffer" and not final_round(state):
                keys.add("product_price")
    elif action_type == "seller_message":
        keys = {"message"}
    else:
        keys = {"decision"}
    if (family in {"bargaining", "negotiation"} and
            state.get("messages_allowed")):
        keys.add("message")
    return keys

def validate_action(game, action):
    if not isinstance(action, dict):
        raise ValueError("strategy must return a dict")
    family = game["game_family"]
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    allowed = contract_action_keys(game, action)
    if set(action) - allowed:
        raise ValueError(f"unexpected action keys: {sorted(set(action) - allowed)}")
    declared = game["valid_actions"].get("fields")
    if isinstance(declared, dict) and declared:
        undeclared = set(action) - set(declared)
        if undeclared:
            raise ValueError(f"keys absent from valid_actions.fields: {sorted(undeclared)}")
    if "message" in action and not isinstance(action["message"], str):
        raise ValueError("message must be a string")
    if family == "bargaining" and action_type == "offer":
        if not is_finite_number(action.get("alice_gain")) or not is_finite_number(action.get("bob_gain")):
            raise ValueError("bargaining gains must be finite numbers")
        alice = float(action["alice_gain"])
        bob = float(action["bob_gain"])
        pot = finite_float(state["money_to_divide"])
        if not all(math.isfinite(x) and x >= 0 for x in (alice, bob)):
            raise ValueError("invalid bargaining allocation")
        if not math.isclose(alice + bob, pot, rel_tol=1e-10, abs_tol=1e-7):
            raise ValueError("bargaining gains do not sum to the pot")
    elif family == "bargaining":
        if action.get("decision") not in {"accept", "reject", "walkaway"}:
            raise ValueError("invalid bargaining decision")
    elif family == "negotiation" and action_type == "offer":
        if (not is_finite_number(action.get("product_price")) or
                float(action["product_price"]) < 0):
            raise ValueError("invalid negotiation price")
    elif family == "negotiation":
        if action.get("decision") not in {"AcceptOffer", "RejectOffer", "WalkAway"}:
            raise ValueError("invalid negotiation decision")
        if action["decision"] == "RejectOffer" and not final_round(state):
            if (not is_finite_number(action.get("product_price")) or
                    float(action["product_price"]) < 0):
                raise ValueError("counteroffer required")
    elif action_type == "seller_message":
        if not isinstance(action.get("message"), str) or len(action["message"]) > 2000:
            raise ValueError("invalid persuasion message")
    elif action.get("decision") not in {"yes", "no"}:
        raise ValueError("invalid persuasion decision")
    if "message" in action and len(action["message"]) > 2000:
        raise ValueError("message exceeds 2,000 characters")
    return action

def strategy(game):
    error = None
    try:
        family = game["game_family"]
        action = validate_action(game, STRATEGIES[family](game))
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        action = validate_action(game, fallback_action(game))
        print(f"SAFE FALLBACK {game.get('game_id')}: {error}")
    with LOCK:
        DECISION_LOG.append({
            "game_id": game.get("game_id"), "family": game.get("game_family"),
            "round": (game.get("game_state") or {}).get("round"),
            "action": dict(action), "error": error,
        })
    return action



## Offline contract, baseline, and portfolio tests

These tests exercise both player roles, extreme bargaining discounts, negotiation feasibility, persuasion observation censoring, candidate-arm isolation, completed-game reward credit, and strict action validation. Test game identifiers stay on protected arms unless a candidate is explicitly forced.


In [ ]:
def base_game(family, action_type, state, player="player_1", game_id="test",
              opponent=None):
    return {
        "game_id": game_id, "game_family": family, "your_player": player,
        "opponent": opponent or {"type": "hidden", "name": None},
        "valid_actions": {"type": action_type, "fields": {}},
        "game_state": state,
    }

def force_arm(game, role, arm):
    POLICY_ASSIGNMENTS[str(game["game_id"])] = {
        "family": game["game_family"], "role": role,
        "context": policy_context(game, role), "arm": arm,
        "player": canonical_player(game["your_player"]),
        "state": dict(game["game_state"]),
    }

def run_v7_tests():
    template = {"current_player": "player_1", "round": 1, "max_rounds": 5,
                "horizon_known": True, "money_to_divide": 100,
                "delta_1": 0.9, "delta_2": 0.95,
                "complete_information": True, "history": [],
                "messages_allowed": True}
    for player in ("player_1", "player_2"):
        state = dict(template, current_player=player)
        game = base_game("bargaining", "offer", state, player, f"v7-b-{player}")
        action = strategy(game)
        assert math.isclose(action["alice_gain"] + action["bob_gain"], 100)
        assert POLICY_ASSIGNMENTS[game["game_id"]]["arm"] == "v4_safe"

    extreme = dict(template, max_rounds=0, horizon_known=False,
                   delta_1=0.9, delta_2=0.999)
    extreme_game = base_game("bargaining", "offer", extreme,
                             "player_1", "v7-b-extreme")
    assert strategy(extreme_game)["bob_gain"] > 90

    # Candidate assignment is stable throughout a game and remains feasible.
    candidate_game = base_game("bargaining", "offer", template,
                               "player_1", "candidate-b")
    force_arm(candidate_game, "player_1", "qre_adaptive")
    first = strategy(candidate_game)
    second = strategy(candidate_game)
    assert first == second and math.isclose(first["alice_gain"] + first["bob_gain"], 100)

    for seller_value, buyer_value in ((0, 100), (40, 100), (99, 100)):
        for player in ("player_1", "player_2"):
            role = "seller" if player == "player_1" else "buyer"
            state = {"current_player": player, "player_1_role": "seller",
                     "player_2_role": "buyer", "player_1_value": seller_value,
                     "player_2_value": buyer_value, "complete_information": True,
                     "round": 1, "max_rounds": 5, "horizon_known": True,
                     "history": [], "messages_allowed": False}
            game = base_game("negotiation", "offer", state, player,
                             f"v7-n-{seller_value}-{buyer_value}-{player}")
            action = strategy(game)
            assert seller_value <= action["product_price"] <= buyer_value
            assert POLICY_ASSIGNMENTS[game["game_id"]]["arm"] == "v5_safe"
            if role == "buyer":
                challenger = base_game("negotiation", "offer", state, player,
                                       f"candidate-n-{seller_value}-{buyer_value}")
                force_arm(challenger, role, "buyer_soft")
                c_action = strategy(challenger)
                assert seller_value <= c_action["product_price"] <= buyer_value

    high = {"current_quality": "high", "product_price": 50, "p": 0.5,
            "v": 100, "u": 0, "round": 1, "total_rounds": 10, "history": []}
    high_game = base_game("persuasion", "seller_recommendation", high,
                          "player_1", "v7-p-high")
    assert strategy(high_game) == {"decision": "yes"}
    assert POLICY_ASSIGNMENTS[high_game["game_id"]]["arm"] == "v3_safe"

    expensive = {"seller_message": "I recommend buying this product.",
                 "product_price": 101, "p": 0.9, "v": 100, "u": 0,
                 "round": 1, "total_rounds": 5, "history": []}
    assert strategy(base_game("persuasion", "buyer_decision", expensive,
                              "player_2", "v7-p-expensive")) == {"decision": "no"}

    # Passed quality is not incorporated into the buyer's reliability model.
    named = {"type": "agent", "name": "audit-opponent"}
    passed = {"seller_message": {"decision": "yes"}, "product_price": 50,
              "p": 0.5, "v": 100, "u": 0, "round": 2, "total_rounds": 5,
              "history": [{"round": 1, "seller_message": {"decision": "yes"},
                           "buyer_decision": "no", "bought": False, "quality": "low"}]}
    pass_game = base_game("persuasion", "buyer_decision", passed,
                          "player_2", "v7-p-pass", named)
    strategy(pass_game)
    pass_key = persuasion_profile_key(pass_game, False)
    assert PERSUASION_MEMORY[pass_key]["pos_low"] == 0

    # Reward credit is arm- and context-specific and is idempotent.
    reward_game = base_game("bargaining", "offer", template,
                            "player_1", "live-reward-test")
    force_arm(reward_game, "player_1", "qre_adaptive")
    assignment = POLICY_ASSIGNMENTS[reward_game["game_id"]]
    reward = normalized_terminal_reward(assignment, {"player_1_payoff": 60})
    assert math.isclose(reward, 0.6)

    assert signal_polarity("Buying would be foolish; do not buy.") is False
    assert signal_polarity("My recommendation: BUY this product.") is True
    invalid = base_game("bargaining", "offer", template, "player_1", "bad-bool")
    try:
        validate_action(invalid, {"alice_gain": True, "bob_gain": 99})
        raise AssertionError("boolean gain was accepted")
    except ValueError:
        pass
    errors = [entry for entry in DECISION_LOG if entry["error"]]
    assert not errors, errors
    print("All V7 protected-baseline, candidate, reward, and contract tests passed.")

run_v7_tests()


## Controlled live evaluation — one family and one role audit at a time

Start with a small batch. V7 explores challengers in at most 12% of under-sampled games and harvests terminal payoff after the run when the SDK exposes it. Run separate cells for each family; do not pool conclusions across roles or configurations. Persuasion should be tested most cautiously because V6's seller losses were concentrated and the aggregate rating fell.


In [ ]:
from glee_sdk import GleeClient

EVALUATION_FAMILY = "bargaining"  # then "negotiation"; test "persuasion" last
CONCURRENCY = 1
MAX_GAMES = 12
MAX_TIME = 3600

client = GleeClient(api_key=os.environ["GLEE_API_KEY"])
before = client.stats()
print("Before:", before)
client.run(
    strategy,
    game_families=[EVALUATION_FAMILY],
    concurrency=CONCURRENCY,
    max_games=MAX_GAMES,
    max_time=MAX_TIME,
)
after = client.stats()
print("After:", after)
harvested = harvest_completed_games(client)
print("Completed-game rewards harvested:", len(harvested))
display(harvested[-20:])


## Inspect fallbacks, assignments, and conservative evidence

Do not infer learning from the number of stored profiles. Inspect which arm actually played, whether terminal rewards were harvested, and the role/configuration-local sample sizes and confidence bounds. A zero harvested count means the installed SDK response did not expose a recognized payoff field; V7 then keeps protected baselines rather than promoting on fabricated evidence.


In [ ]:
fallbacks = [entry for entry in DECISION_LOG if entry["error"]]
print("Fallback count:", len(fallbacks))
print("Recent decisions:")
display(list(DECISION_LOG)[-20:])

print("Recent policy assignments:")
display(list(POLICY_ASSIGNMENTS.items())[-30:])

evidence = []
for key in sorted(POLICY_STATS, key=str):
    n, mean, lower, upper = arm_summary(key)
    evidence.append({"family": key[0], "context": key[1], "arm": key[2],
                     "n": n, "mean_reward": round(mean, 4),
                     "lower": round(lower, 4), "upper": round(upper, 4)})
print("Completed-game policy evidence:")
display(evidence)

print("Named/game-local bargaining profiles:", len(BARGAINING_MEMORY))
print("Named/game-local negotiation profiles:", len(NEGOTIATION_MEMORY))
print("Role-separated persuasion profiles:", len(PERSUASION_MEMORY))
